# Generated UUID for new items in catalog

This notebook demoes the story RSPY-1239: https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1239

This demo proceeds in 7 steps:
1. Insert a new item in the catalog
2. Retrieve the item and get its UUID
3. Send a PUT request to update the item with a different UUID
4. Retrieve the item and check that the UUID HAS NOT changed
5. Send a PATCH request to update the item with a different UUID
6. Retrieve and check the item's UUID again, it must stay with the original value
7. Retrieve the item using an Odata request and check that it also contains the UUID

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *
import pprint
init_demo()
# Reload the global vars again
from resources.utils import *  

from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)

## Stage an item in the catalog

In [ ]:
# Create a test collection 
CATALOG_COLLECTION_ID = "SPRINT41_1239_ITEMS_UUID"
collection = create_test_collection(CATALOG_COLLECTION_ID)
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
list(items)

In [ ]:
# Stage an Auxip item in the new collection of the catalog
item_id = "S1A_AUX_PP2_V20200920T080000_G20200920T080000"
stac_filter = f"id in ('{item_id}')"
item_collection_auxip = auxip_client.search(
    method="GET",
    stac_filter=stac_filter
)

staged_item = stage_data(item_collection_auxip.to_dict(), items_id = [], catalog_collection_name = CATALOG_COLLECTION_ID)

In [ ]:
# Display the UUID automatically created for this new item
original_catalog_item = staged_item[0].to_dict()

for ext_id in original_catalog_item["properties"]["externalIds"]:
    if ext_id["scheme"] == "rs-odata-prip":
        original_item_uuid = ext_id["value"]
print("========== ORIGINAL UUID OF INSERTED ITEM ==========")
print(original_item_uuid)

## Test that the UUID cannot be patched

In [ ]:
# Update 2 fields of the item: the UUID and a random field to make sure the request was actually processed
updated_item = original_catalog_item.copy()
updated_item["properties"]["platform"] = "toulouse_matabiau"
for ext_id in updated_item["properties"]["externalIds"]:
    if ext_id["scheme"] == "rs-odata-prip":
        ext_id["value"] = "123456789"
print("===== PATCHED ITEM SENT =====")
display(Item.from_dict(updated_item))

# Send update request
catalog_client.update_item(updated_item)

# Retrieve the item from the catalog
updated_catalog_item = catalog_client.get_item(CATALOG_COLLECTION_ID, item_id, OWNER_ID).to_dict()
updated_platform = updated_catalog_item["properties"]["platform"]
for ext_id in updated_catalog_item["properties"]["externalIds"]:
    if ext_id["scheme"] == "rs-odata-prip":
        updated_uuid = ext_id["value"]

# Show the patched item
print("===== PATCHED ITEM IN THE CATALOG =====")
display(Item.from_dict(updated_catalog_item))

# Check that one value has changed but not the other
assert updated_platform == "toulouse_matabiau"
assert updated_uuid == original_item_uuid

In [ ]:
# Patch 2 fields of the item: the same "externalIds" (here taken from the previous updat_item) and another different field
externalids_to_patch = updated_item["properties"]["externalIds"]
constellation_to_patch = "orion"

# Send PATCH request
catalog_client.patch_item(
    collection_id=CATALOG_COLLECTION_ID,
    item_id=item_id,
    patch_values={
        "properties": {
            "externalIds":externalids_to_patch,
            "constellation":constellation_to_patch
        }
    },
    owner_id=OWNER_ID
)

# Retrieve the item from the catalog
updated_catalog_item = catalog_client.get_item(CATALOG_COLLECTION_ID, item_id, OWNER_ID).to_dict()
updated_constellation = updated_catalog_item["properties"]["constellation"]
for ext_id in updated_catalog_item["properties"]["externalIds"]:
    if ext_id["scheme"] == "rs-odata-prip":
        updated_uuid = ext_id["value"]

# Show the patched item
print("===== PATCHED ITEM IN THE CATALOG =====")
display(Item.from_dict(updated_catalog_item))

# Check that one value has changed but not the other
assert updated_constellation == "orion"
assert updated_uuid == original_item_uuid

## Retrieve the element with an Odata request

In [ ]:
# Define 2 dates to filter data on
d1 = original_catalog_item["properties"]["published"]
d2 = original_catalog_item["properties"]["published"]

# Endpoint to request
openapi_endpoint = f"/catalog/odata/v1/Products?$filter=PublicationDate gt {d1} and PublicationDate lt {d2}"

# Send request
url = f"{catalog_client.href_service}{openapi_endpoint}"
headers = {'Accept': 'application/json',
          "X-API-Key": os.getenv("RSPY_APIKEY")}

openapi_response = requests.get(url, headers=headers)
odata_items = openapi_response.json()
display(odata_items)

# Find item and check that the UUID is correct
for item in odata_items:
    if item["Name"]==item_id:
        odata_item_uuid = item['Id']
        assert item['Id'] == original_item_uuid